In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Model 2

### State Variables

All three variables are totals for Gilbert Bay (the south arm), not concentrations.

| Symbol | Meaning | Units |
|---|---|---|
| $S(t)$ | Total brine shrimp | shrimp |
| $A(t)$ | Total algae | µg Chl-a |
| $B(t)$ | Total birds | birds |

### Equations

$$
\frac{dS}{dt} = \gamma_S C_A S A - C_S S B - \delta_S S
$$

$$
\frac{dA}{dt} = \beta_A A(1-A/K) - C_A S A
$$

$$
\frac{dB}{dt} = -\delta_B B + \gamma_B C_S S B
$$

### Parameters

| Parameter | Meaning | Value | Units | Brief justification |
|:---:|---|---|---|---|
| $V$ | Lake volume | 2.25 × 10¹³ | L | Volume of the whole lake (22.5 km ^ 3 in liters), from a source that peter got. |
| $\beta_A$ | Algae growth rate | 0.3 | day⁻¹ | A guess. This means that in a sparse environment with only a tiny bit of algae, we could expect it to double every few days, which is realistic. |
| $K$ | Algae carrying capacity (whole lake) | 6 × 10¹⁴ | µg Chl-a | A guess of 50 µg/L × V.  Some studies have used similar numbers. This is the most algae the lake could hold if nothing ate it; growth slows to zero as algae approach this level. |
| $C_A$ | Shrimp feeding rate | 1.7 × 10⁻¹⁵ | 1/(shrimp·day) | A guess.  One shrimp filters about 0.02 L of water per day (Makridis & Vadstein, 1999). Divided by V, this is the tiny fraction of the lake one shrimp clears each day. |
| $C_S$ | Bird feeding rate | 5.8 × 10⁻⁹ | 1/(bird·day) | A guess.  If one grebe searches about 70,000 L of water per day then, dividing by V, this is the fraction of the lake one bird searches. |
| $\gamma_S$ | Shrimp gained per algae eaten | 0.2 | shrimp/µg Chl-a | A guess. This means shrimp must eat about 5 µg of algae (chlorophyll-a) to produce one new shrimp. |
| $\gamma_B$ | Birds gained per shrimp eaten | 5 × 10⁻⁸ | bird/shrimp | A guess. A bird eats ~9 million shrimp per year and raises ~0.5 offspring per year, so it takes ~20 million shrimp to make one new bird. |
| $\delta_S$ | Shrimp natural mortality | 0.03 | day⁻¹ | A guess. This means shrimp live about a month on average (1/0.03 ≈ 33 days), which fits brine shrimp dying off within a season. |
| $\delta_B$ | Bird natural mortality | 0.0003 | day⁻¹ | A guess.  Grebes live around 5 to 10 years.  Choosing an average of 7-8 years = 3000 days ≈ 1 / 0.0003.

In [ ]:
# Parameters:
V = 2.25 * 10 ** 13
beta_A = 0.3
K = 6 * 10 ** 14
C_A = 1.7 * 10 ** -15
C_S = 5.8 * 10 ** -9
gamma_S = 0.2
gamma_B = 5 * 10 ** -8
delta_S = 0.03
delta_B = 0.0003

# Model 1

In [ ]:
def model(t, y, delta_B):
    S, A, B = y
    dS = gamma_S * C_A * S * A - C_S * S * B
    dA = beta_A * A - C_A * S * A
    dB = -delta_B * B + gamma_B * C_S * S * B
    return [dS, dA, dB]


def run(y0, delta_B, T):
    t = np.linspace(0, T, 3000)
    sol = solve_ivp(model, (0, T), y0, args=(delta_B,), t_eval=t,
                    method="LSODA", rtol=1e-9, atol=1e-12)
    return sol.t, sol.y

# Plot 1: shrimp vs algae with no birds (B = 0)
S_grid, A_grid = np.meshgrid(np.linspace(0.01, 6, 40), np.linspace(0.01, 12, 40))
dS_grid = gamma_S * C_A * S_grid * A_grid
dA_grid = beta_A * A_grid - C_A * S_grid * A_grid

plt.figure(figsize=(7, 5.5))
plt.streamplot(S_grid, A_grid, dS_grid, dA_grid, color="lightgray", density=1.1)

for S0, A0 in [(0.5, 1), (1, 1), (1.5, 1), (0.6, 3)]:
    t, y = run([S0, A0, 0.0], delta_B=0.5, T=40)   # delta_B unused when B = 0
    line, = plt.plot(y[0], y[1], linewidth=2, label=f"start S={S0}, A={A0}")
    plt.plot(S0, A0, "o", color=line.get_color())
    plt.legend()


plt.axvline(beta_A / C_A, color="gray", linestyle="--", label="algae nullcline S = β_A/C_A")
plt.xlim(0, 6)
plt.ylim(0, 12)
plt.xlabel("Shrimp S")
plt.ylabel("Algae A")
plt.title("Shrimp–algae phase plane, no birds (B = 0)")
plt.tight_layout()
plt.show()
plt.savefig("shrimp_algae_plane.png", dpi=600)






# ------


# Plot 2: time series for three bird death rates

balanced = gamma_B * C_S * beta_A / C_A   # = 0.24, the only value allowing cycles
cases = [("Birds die faster than balance", 0.50),
         ("Balanced (knife-edge)", balanced),
         ("Birds die slower than balance", 0.15)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (name, delta_B) in zip(axes, cases):
    t, y = run([1.0, 1.0, 1.0], delta_B, T=80)
    ax.plot(t, y[0], label="Shrimp S")
    ax.plot(t, y[1], label="Algae A")
    ax.plot(t, y[2], label="Birds B")
    ax.set_yscale("log")
    ax.set_ylim(1e-6, 1e6)
    ax.set_title(f"{name}\nδ_B = {delta_B:.2f}")
    ax.set_xlabel("Time")
    ax.legend()
axes[0].set_ylabel("Population (log scale)")
plt.tight_layout()
plt.savefig("timeseries.png", dpi=600)

plt.show()